### Question 3.6 answer: ResNeXt

For the Kaggle competition we implement **ResNeXt** from the original paper:

**> S. Xie, R. Girshick, P. Dollár, Z. Tu, K. He. *Aggregated Residual Transformations for Deep Neural Networks*. CVPR 2017. [arXiv:1611.05431](https://arxiv.org/abs/1611.05431)

**Main idea of the paper.** Besides depth and width, ResNeXt introduces a third dimension, the **cardinality** $C$: the number of parallel transformations aggregated inside a residual block. A ResNeXt block computes (Eq. 2 of the paper)**

$$y = x + \sum_{i=1}^{C} \mathcal{T}_i(x)$$

where every $\mathcal{T}_i$ has the same *bottleneck* topology (a $1\times1$ conv that reduces the channels to a small width $d$, a $3\times3$ conv, and a $1\times1$ conv that expands the channels back). Figure 3 of the paper shows three equivalent ways to implement this block; form **(c)** merges the $C$ parallel paths into a single **grouped convolution** with `groups = C` (a $1\times1$ conv from the input to $C\cdot d$ channels, a $3\times3$ grouped conv with $C\cdot d$ channels, and a $1\times1$ conv back to the block width), which is what we implement below. The paper shows that, for the same number of parameters and FLOPs, increasing the cardinality is more effective than going deeper or wider (e.g. ResNeXt-50 32×4d beats ResNet-50, and on CIFAR the test error drops from 1×64d to 16×64d, Sec. 5.3 and Fig. 7).

**Architecture used here.** We follow the CIFAR template of Sec. 5.3 of the paper (ResNeXt-29): a single $3\times3$ conv stem, then **three stages of three ResNeXt blocks**, where the feature map is down-sampled (stride 2) and the group width $d$ is doubled at the start of stages 2 and 3, followed by **global average pooling** and a fully connected layer. Depth $= 9 \cdot 3 + 2 = 29$ layers. Because our dataset is small (about 9,000 training images of $64\times64$ pixels) we use cardinality $C = 8$ with group width $d = 8$ (a *ResNeXt-29 (8×8d)*) and halve the block output widths of the paper (128, 256, 512 instead of 256, 512, 1024). The shortcut is the identity when the shapes match, and a $1\times1$ conv (stride 2) + batch norm projection otherwise (Sec. 3.1 / ResNet option B).

**Initialisation.** All convolutional and linear layers use **He initialisation** (He et al., *Delving Deep into Rectifiers*, ICCV 2015): weights drawn from $\mathcal{N}(0, 2/n)$ where $n$ is the fan of the layer, which keeps the variance of the activations constant through the ReLU layers of a deep network; biases are zero and batch-norm layers start at $\gamma=1,\beta=0$.

**Training recipe.** As in the CIFAR experiments of the paper we train with SGD (momentum 0.9, weight decay $5\cdot10^{-4}$), a mini-batch size of 64, and standard augmentation (random crop with 4-pixel padding and random horizontal flip). The learning rate starts at 0.05 and follows a cosine schedule instead of the two step decays used in the paper. Two models are trained with this recipe: one without data mixing and one with **Mixup + CutMix** (Questions 3.2 and 3.3): for every mini-batch, with probability 0.8, either Mixup ($\alpha=0.4$) or CutMix ($\alpha=1$) is applied. The model with the best validation accuracy is kept for the Kaggle submission.

In [ ]:
#Your code here
# ResNeXt - Xie et al., "Aggregated Residual Transformations for Deep Neural Networks", CVPR 2017
# https://arxiv.org/abs/1611.05431
import torch.nn.functional as F


def he_initialize(model):
    """
    He (Kaiming) initialisation - He, Zhang, Ren, Sun, "Delving Deep into Rectifiers: Surpassing Human-Level
    Performance on ImageNet Classification", ICCV 2015, https://arxiv.org/abs/1502.01852.
    For a layer with n inputs (fan) followed by a ReLU, drawing the weights from N(0, 2/n) (Eq. 10 of the paper)
    keeps the variance of the activations constant through the network, so that very deep ReLU networks can
    be trained from scratch (Xavier initialisation, which assumes a linear activation, halves the variance at
    every ReLU layer and makes the signal vanish in deep networks).
      * conv layers  : N(0, 2 / (k*k*C_out))   ('fan_out' mode, the variant used by the official ResNet/ResNeXt code
                                                 that preserves the variance of the gradients in the backward pass)
      * linear layer : N(0, 2 / n_in)          ('fan_in' mode)
      * biases       : 0
      * batch norm   : gamma = 1, beta = 0
    """
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


class ResNeXtBlock(nn.Module):
    """
    One ResNeXt bottleneck block, implemented as form (c) of Fig. 3 of the paper:

        x --> 1x1 conv (in -> C*d) -> BN -> ReLU
          --> 3x3 GROUPED conv (C*d -> C*d, groups = C, stride s) -> BN -> ReLU
          --> 1x1 conv (C*d -> out) -> BN
          --> + shortcut(x) --> ReLU

    The grouped convolution is exactly the aggregation of C transformations of width d,
    y = x + sum_{i=1..C} T_i(x)  (Eq. 2 of the paper). The shortcut is the identity when the
    input and output shapes match and a 1x1 conv (stride s) + BN projection otherwise.
    """
    def __init__(self, in_channels, out_channels, stride=1, cardinality=8, group_width=8):
        super().__init__()
        width = cardinality * group_width                      # C * d channels inside the block
        self.conv_reduce = nn.Conv2d(in_channels, width, kernel_size=1, bias=False)
        self.bn_reduce = nn.BatchNorm2d(width)
        self.conv_group = nn.Conv2d(width, width, kernel_size=3, stride=stride, padding=1,
                                    groups=cardinality, bias=False)   # <- cardinality = number of groups
        self.bn_group = nn.BatchNorm2d(width)
        self.conv_expand = nn.Conv2d(width, out_channels, kernel_size=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        out = F.relu(self.bn_reduce(self.conv_reduce(x)))
        out = F.relu(self.bn_group(self.conv_group(out)))
        out = self.bn_expand(self.conv_expand(out))
        return F.relu(out + self.shortcut(x))


class ResNeXt(nn.Module):
    """
    .3 of the paper:
ResNeXt for small images, following the CIFAR template of Sec. 5
        3x3 conv stem -> stage 1 (n blocks, stride 1) -> stage 2 (n blocks, stride 2)
                      -> stage 3 (n blocks, stride 2) -> global average pooling -> FC

    The group width d is doubled every time the feature map is down-sampled (the paper's rule to
    keep the complexity of each stage roughly constant), so the 3x3 grouped conv has
    C*d, 2*C*d and 4*C*d channels in the three stages. The depth is 9*n + 2 (n = 3 -> ResNeXt-29).

    cardinality   : C, the number of groups / aggregated transformations per block
    group_width   : d, the width of each transformation in the first stage
    stage_channels: output width of the blocks in each stage
    num_blocks    : number of blocks per stage
    """
    def __init__(self, num_classes=20, cardinality=8, group_width=8, stage_channels=(128, 256, 512),
                 num_blocks=(3, 3, 3), stem_channels=64, drop_rate=0.0):
        super().__init__()
        self.config = dict(num_classes=num_classes, cardinality=cardinality, group_width=group_width,
                           stage_channels=tuple(stage_channels), num_blocks=tuple(num_blocks),
                           stem_channels=stem_channels, drop_rate=drop_rate)   # saved with the checkpoints
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True))

        stages, in_channels, d = [], stem_channels, group_width
        for stage_index, (out_channels, n) in enumerate(zip(stage_channels, num_blocks)):
            stride = 1 if stage_index == 0 else 2          # down-sample at the start of stages 2 and 3
            blocks = []
            for block_index in range(n):
                blocks.append(ResNeXtBlock(in_channels, out_channels,
                                           stride=stride if block_index == 0 else 1,
                                           cardinality=cardinality, group_width=d))
                in_channels = out_channels
            stages.append(nn.Sequential(*blocks))
            d *= 2                                          # double the group width for the next stage
        self.stages = nn.Sequential(*stages)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))            # global average pooling
        self.dropout = nn.Dropout(drop_rate)
        self.fc = nn.Linear(in_channels, num_classes)

        he_initialize(self)     # He initialisation of all layers (see he_initialize above)

    def features(self, x):
        """Penultimate representation (output of the global average pooling)."""
        x = self.stages(self.stem(x))
        return torch.flatten(self.pool(x), 1)

    def forward(self, x):
        return self.fc(self.dropout(self.features(x)))


resnext = ResNeXt(num_classes=len(class_names), cardinality=8, group_width=8,
                  stage_channels=(128, 256, 512), num_blocks=(3, 3, 3), drop_rate=0.2).to(device)
print(resnext)
n_params = sum(p.numel() for p in resnext.parameters())
print(f"\nResNeXt-29 (8x8d): {n_params/1e6:.2f}M parameters")
with torch.no_grad():
    print("output shape for a [2, 3, 64, 64] batch:", tuple(resnext(torch.zeros(2, 3, 64, 64, device=device)).shape))

# check of the He initialisation: the empirical std of a conv weight must match sqrt(2 / fan_out)
conv = resnext.stages[0][0].conv_group
fan_out = conv.out_channels * conv.kernel_size[0] * conv.kernel_size[1]     # PyTorch's fan_out = C_out * k * k
print(f"He init check on a 3x3 grouped conv: weight std = {conv.weight.std().item():.4f}, sqrt(2/fan_out) = {(2 / fan_out) ** 0.5:.4f}")

In [ ]:
# Data pipeline and training loop for ResNeXt.
# Augmentation as in the CIFAR experiments of the paper: random crop with 4-pixel zero padding
# and random horizontal flip (plus the colour jitter already used above). The validation images
# are NOT augmented, so we build two ImageFolder views of the dataset and split them with the same
# random permutation.
from torch.utils.data import Subset

resnext_train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
resnext_eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset_aug = datasets.ImageFolder(data_dir, transform=resnext_train_transform)
dataset_plain = datasets.ImageFolder(data_dir, transform=resnext_eval_transform)
assert len(dataset_aug.classes) == len(class_names), dataset_aug.classes
print("class folders:", dataset_aug.classes)     # same alphabetical order as class_names (folder names are plural)

perm = torch.randperm(len(dataset_aug), generator=torch.Generator().manual_seed(1234))
n_train = int(0.9 * len(dataset_aug))
train_idx, val_idx = perm[:n_train].tolist(), perm[n_train:].tolist()

num_workers = 2 if os.name != 'nt' else 0      # worker processes are not needed on Windows/Colab-CPU
resnext_train_loader = DataLoader(Subset(dataset_aug, train_idx), batch_size=64, shuffle=True,
                                  num_workers=num_workers, pin_memory=train_on_gpu, drop_last=True)
resnext_val_loader = DataLoader(Subset(dataset_plain, val_idx), batch_size=128, shuffle=False,
                                num_workers=num_workers, pin_memory=train_on_gpu)
print(f"ResNeXt loaders: {len(train_idx)} training images, {len(val_idx)} validation images")


# ---- Mixup / CutMix for the ResNeXt training (same definitions as in Questions 3.2 and 3.3) --------------
def mix_batch(X, y, mode, mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.8):
    """
    Returns (X_mixed, y_a, y_b, lam) so that the loss is  lam * CE(f(X_mixed), y_a) + (1 - lam) * CE(f(X_mixed), y_b).
    mode: None (no mixing), 'mixup', 'cutmix' or 'mixup+cutmix' (one of the two is drawn at random for every batch).
    With probability 1 - mix_prob the batch is left clean, so the model also sees un-mixed images.
    """
    if mode is None or np.random.rand() > mix_prob:
        return X, y, y, 1.0
    if mode == 'mixup+cutmix':
        mode = 'mixup' if np.random.rand() < 0.5 else 'cutmix'
    index = torch.randperm(X.size(0), device=X.device)
    if mode == 'mixup':                      # Zhang et al. 2018: convex combination of two images
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        return lam * X + (1 - lam) * X[index], y, y[index], lam
    if mode == 'cutmix':                     # Yun et al. 2019: paste a random box of image B into image A
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        H, W = X.size(2), X.size(3)
        cut_h, cut_w = int(H * np.sqrt(1 - lam)), int(W * np.sqrt(1 - lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = np.clip(cy - cut_h // 2, 0, H), np.clip(cy + cut_h // 2, 0, H)
        x1, x2 = np.clip(cx - cut_w // 2, 0, W), np.clip(cx + cut_w // 2, 0, W)
        X_cut = X.clone()
        X_cut[:, :, y1:y2, x1:x2] = X[index, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)          # exact area ratio after clipping
        return X_cut, y, y[index], lam
    raise ValueError(mode)


@torch.no_grad()
def evaluate(model, loader, loss_fn=nn.CrossEntropyLoss()):
    """Average loss and accuracy of `model` on `loader`."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


def fit_resnext(model, train_loader, val_loader, num_epochs=40, lr=0.05, momentum=0.9,
                weight_decay=5e-4, seed=1234, mix=None, checkpoint_path=None):
    """
    SGD with momentum and weight decay as in the paper (Sec. 5.3), with a cosine learning-rate
    schedule. `mix` = None / 'mixup' / 'cutmix' / 'mixup+cutmix' selects the data mixing applied to
    every training batch. Keeps the weights of the epoch with the best validation accuracy and, if
    `checkpoint_path` is given, writes them to that file every time the validation accuracy improves.
    (With mixing, the reported training accuracy is the lam-weighted accuracy on the mixed labels.)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum,
                                weight_decay=weight_decay, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_state = 0.0, None

    for epoch in range(num_epochs):
        model.train()
        t0 = time.time()
        running_loss, running_correct, seen = 0.0, 0.0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            X, y_a, y_b, lam = mix_batch(X, y, mix)
            optimizer.zero_grad()
            logits = model(X)
            loss = lam * loss_fn(logits, y_a) + (1 - lam) * loss_fn(logits, y_b)
            loss.backward()
            optimizer.step()
            pred = logits.argmax(1)
            running_loss += loss.item() * y.size(0)
            running_correct += lam * (pred == y_a).sum().item() + (1 - lam) * (pred == y_b).sum().item()
            seen += y.size(0)
        scheduler.step()

        train_loss, train_acc = running_loss / seen, running_correct / seen   # running metrics of the epoch
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        for k, v in zip(history, (train_loss, train_acc, val_loss, val_acc)):
            history[k].append(v)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            if checkpoint_path is not None:
                os.makedirs(os.path.dirname(checkpoint_path) or '.', exist_ok=True)
                torch.save({'state_dict': best_state, 'config': model.config, 'epoch': epoch + 1,
                            'val_acc': val_acc, 'class_names': class_names}, checkpoint_path)
        print(f"Epoch {epoch+1}/{num_epochs} ({time.time()-t0:.0f}s, lr={scheduler.get_last_lr()[0]:.4f}) "
              f"train loss= {train_loss:.4f} - train acc= {train_acc*100:.2f}% - "
              f"valid loss= {val_loss:.4f} - valid acc= {val_acc*100:.2f}%")

    model.load_state_dict(best_state)
    print(f"Best validation accuracy: {best_acc*100:.2f}%")
    return history


# (a) ResNeXt-29 (8x8d) with He initialisation, trained WITHOUT data mixing (reference run)
resnext_history = fit_resnext(resnext, resnext_train_loader, resnext_val_loader, num_epochs=40, lr=0.05, mix=None,
                              checkpoint_path='models/resnext29_8x8d_plain_40ep.pt')

val_loss, val_acc = evaluate(resnext, resnext_val_loader)
print(f"ResNeXt-29 (8x8d), no mixing, best checkpoint - valid loss= {val_loss:.4f} - valid acc= {val_acc*100:.2f}%")

In [ ]:
# (b) The same ResNeXt-29 (8x8d), He-initialised, trained WITH Mixup + CutMix.
# For every mini-batch (with probability 0.8) either Mixup (alpha = 0.4) or CutMix (alpha = 1.0) is applied,
# each with probability 1/2; the remaining 20% of the batches are clean. Mixing the two strategies is the
# recipe used by modern ImageNet training pipelines (e.g. DeiT, timm) and combines the strengths of both:
# Mixup smooths the decision boundaries between classes while CutMix forces the model to use several local
# regions of the object and keeps the training pixels natural.
resnext_mix = ResNeXt(num_classes=len(class_names), cardinality=8, group_width=8,
                      stage_channels=(128, 256, 512), num_blocks=(3, 3, 3), drop_rate=0.2).to(device)
he_initialize(resnext_mix)
resnext_mix_history = fit_resnext(resnext_mix, resnext_train_loader, resnext_val_loader,
                                  num_epochs=40, lr=0.05, mix='mixup+cutmix',
                                  checkpoint_path='models/resnext29_8x8d_mixup_cutmix_40ep.pt')

val_loss_mix, val_acc_mix = evaluate(resnext_mix, resnext_val_loader)
print(f"ResNeXt-29 (8x8d), Mixup + CutMix, best checkpoint - valid loss= {val_loss_mix:.4f} - valid acc= {val_acc_mix*100:.2f}%")
print(f"ResNeXt-29 (8x8d), no mixing,      best checkpoint - valid loss= {val_loss:.4f} - valid acc= {val_acc*100:.2f}%")

# the model used for the Kaggle submission is the one with the best validation accuracy
best_resnext, best_resnext_name = (resnext_mix, "Mixup + CutMix") if val_acc_mix >= val_acc else (resnext, "no mixing")
print(f"\nModel kept for the Kaggle submission: ResNeXt-29 (8x8d) trained with {best_resnext_name}")

# learning curves of the two runs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for h, name, ls in [(resnext_history, 'no mixing', '--'), (resnext_mix_history, 'Mixup + CutMix', '-')]:
    axes[0].plot(h['train_loss'], ls, color='C0', label=f'train ({name})'); axes[0].plot(h['val_loss'], ls, color='C1', label=f'valid ({name})')
    axes[1].plot(h['train_acc'], ls, color='C0', label=f'train ({name})'); axes[1].plot(h['val_acc'], ls, color='C1', label=f'valid ({name})')
axes[0].set_title('ResNeXt loss'); axes[0].set_xlabel('epoch'); axes[0].legend(fontsize=8)
axes[1].set_title('ResNeXt accuracy (train acc. of the mixed run is measured on mixed labels)'); axes[1].set_xlabel('epoch'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# (c) Longer schedule: the Mixup + CutMix recipe with a single cosine schedule over 80 epochs.
# Data mixing slows down the fit of the training set (the mixed run above was still improving at epoch 40),
# so the same model is trained again from scratch (He initialisation) with the cosine schedule stretched to 80 epochs.
resnext_mix80 = ResNeXt(num_classes=len(class_names), cardinality=8, group_width=8,
                        stage_channels=(128, 256, 512), num_blocks=(3, 3, 3), drop_rate=0.2).to(device)
he_initialize(resnext_mix80)
resnext_mix80_history = fit_resnext(resnext_mix80, resnext_train_loader, resnext_val_loader,
                                    num_epochs=80, lr=0.05, mix='mixup+cutmix',
                                    checkpoint_path='models/resnext29_8x8d_mixup_cutmix_80ep.pt')

val_loss_mix80, val_acc_mix80 = evaluate(resnext_mix80, resnext_val_loader)
print()
print(f"ResNeXt-29 (8x8d), no mixing,      40 epochs - valid loss= {val_loss:.4f} - valid acc= {val_acc*100:.2f}%")
print(f"ResNeXt-29 (8x8d), Mixup + CutMix, 40 epochs - valid loss= {val_loss_mix:.4f} - valid acc= {val_acc_mix*100:.2f}%")
print(f"ResNeXt-29 (8x8d), Mixup + CutMix, 80 epochs - valid loss= {val_loss_mix80:.4f} - valid acc= {val_acc_mix80*100:.2f}%")

if val_acc_mix80 >= max(val_acc, val_acc_mix):
    best_resnext, best_resnext_name = resnext_mix80, "Mixup + CutMix, 80 epochs"
print(f"\nModel kept for the Kaggle submission: ResNeXt-29 (8x8d) trained with {best_resnext_name}")

# validation accuracy of the three runs
plt.figure(figsize=(7, 4))
for h, name in [(resnext_history, 'no mixing, 40 epochs'), (resnext_mix_history, 'Mixup + CutMix, 40 epochs'),
                (resnext_mix80_history, 'Mixup + CutMix, 80 epochs')]:
    plt.plot(range(1, len(h['val_acc']) + 1), [a * 100 for a in h['val_acc']], label=name)
plt.xlabel('epoch'); plt.ylabel('validation accuracy (%)'); plt.title('ResNeXt-29 (8x8d)'); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Save the best ResNeXt so that it can be exported / reused (e.g. in the Kaggle notebook).
# The checkpoint holds the weights (state_dict), the constructor arguments needed to rebuild the model
# and the class names, so the file is self-contained: rebuild with  ResNeXt(**ckpt['config'])  and
# load_state_dict(ckpt['state_dict']).  (The training loop above also wrote one checkpoint per run in ./models.)
best_model_path = 'models/resnext29_8x8d_best.pt'
os.makedirs('models', exist_ok=True)
torch.save({'state_dict': best_resnext.state_dict(), 'config': best_resnext.config,
            'class_names': class_names, 'trained_with': best_resnext_name}, best_model_path)
print(f"saved {best_model_path} ({os.path.getsize(best_model_path)/1e6:.1f} MB): ResNeXt-29 (8x8d) trained with {best_resnext_name}")

# Check that the file can be reloaded into a fresh model and gives the same validation accuracy
ckpt = torch.load(best_model_path, map_location=device)
reloaded = ResNeXt(**ckpt['config']).to(device)
reloaded.load_state_dict(ckpt['state_dict'])
reload_loss, reload_acc = evaluate(reloaded, resnext_val_loader)
print(f"reloaded model - valid loss= {reload_loss:.4f} - valid acc= {reload_acc*100:.2f}%")

# The same weights exported to TorchScript, which can be loaded without the Python class definition
# (torch.jit.load('models/resnext29_8x8d_best.torchscript.pt')) and to ONNX for other frameworks.
reloaded.eval()
scripted = torch.jit.trace(reloaded, torch.zeros(1, 3, 64, 64, device=device))
scripted.save('models/resnext29_8x8d_best.torchscript.pt')
try:
    torch.onnx.export(reloaded, torch.zeros(1, 3, 64, 64, device=device), 'models/resnext29_8x8d_best.onnx',
                      input_names=['image'], output_names=['logits'], dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}})
    print("exported models/resnext29_8x8d_best.torchscript.pt and models/resnext29_8x8d_best.onnx")
except Exception as e:      # the onnx package is optional
    print("exported models/resnext29_8x8d_best.torchscript.pt (ONNX export skipped:", str(e)[:80], ")")


In [ ]:
# Predicting the Kaggle test set with the trained ResNeXt.
# The Kaggle test images are unlabeled, so we read them from a folder in sorted file-name order.
# Set `kaggle_test_dir` to the folder produced by the Kaggle starter notebook and adapt the two
# column names of the submission file to its sample_submission.csv if they differ.
from torch.utils.data import Dataset
import csv

kaggle_test_dir = "./FIT5215_Dataset_test"     # <- folder with the Kaggle test images
submission_path = "./resnext_submission.csv"


class UnlabeledImageFolder(Dataset):
    """All images of a folder (recursively), returned as (tensor, file name) in sorted order."""
    def __init__(self, root, transform):
        exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        self.files = sorted(p for p in pathlib.Path(root).rglob('*') if p.suffix.lower() in exts)
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = PIL.Image.open(self.files[i]).convert('RGB')
        return self.transform(img), self.files[i].name


@torch.no_grad()
def predict_folder(model, folder, transform=resnext_eval_transform, batch_size=128, tta=True):
    """
    Returns a list of (file name, predicted class index). With `tta=True` the prediction averages the
    softmax of the image and of its horizontal flip (test-time augmentation).
    """
    model.eval()
    loader = DataLoader(UnlabeledImageFolder(folder, transform), batch_size=batch_size, shuffle=False)
    predictions = []
    for X, names in loader:
        X = X.to(device)
        probs = torch.softmax(model(X), dim=1)
        if tta:
            probs = 0.5 * (probs + torch.softmax(model(torch.flip(X, dims=[3])), dim=1))
        predictions += list(zip(names, probs.argmax(1).tolist()))
    return predictions


if os.path.isdir(kaggle_test_dir):
    predictions = predict_folder(best_resnext, kaggle_test_dir)     # the better of the two ResNeXt runs
    with open(submission_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'label'])                           # adapt to sample_submission.csv if needed
        for name, class_index in predictions:
            writer.writerow([pathlib.Path(name).stem, class_names[class_index]])
    print(f"Wrote {len(predictions)} predictions to {submission_path}")
    print(predictions[:5])
else:
    print(f"Kaggle test folder '{kaggle_test_dir}' not found - download it with the Kaggle starter notebook first.")